# Cross-Tribal and Regional Fire Collaboration Analysis
**Series:** Tribal Fire Science & Indigenous Data Sovereignty  
**Author:** Lilly Jones, PhD  
**Last Updated:** 2025  
**Data Sources:** Census TIGER AIANNH, USGS WBD HUC-8, EPA Level III Ecoregions, MTBS

## Overview
Fire does not respect political boundaries. Effective fire management requires
collaboration across Tribal, federal, state, and local jurisdictions. This notebook
identifies inter-Tribal collaboration opportunities by analyzing shared watersheds,
shared ecoregions, geographic proximity, and overlapping fire histories. A network
graph quantifies connection strength and ranks the highest-priority collaboration pairs.

## Research Questions
- Which Tribal Nations share watersheds, ecoregions, or fire corridors?
- Which collaboration pairs have the highest combined capacity and shared risk?
- Where are natural collaboration clusters based on geography and fire regime?

## Outputs
- Regional collaboration map with connection lines
- Network graph of Tribal collaboration opportunities
- Collaboration opportunity ranking table
- Shared watershed and ecoregion summaries

## Data Sovereignty Note
> This analysis uses federal boundary and ecological datasets to identify
> collaboration opportunities, it does not define Tribal territory or speak
> to Tribal governance structures. Any actual collaboration framework must
> be developed by and with the Tribal Nations involved, on their own terms.
> See the acknowledgment below. No synthetic data is used.

In [1]:
# Imports
import sys
from pathlib import Path

REPO_ROOT = Path().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import warnings
from datetime import datetime

import geopandas as gpd
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.cluster.hierarchy import dendrogram, fcluster, linkage
from scipy.spatial import distance_matrix
from shapely.geometry import LineString
from shapely.ops import unary_union
from shapely.validation import make_valid

from src.data import constants, loaders, validators
from src.data.constants import PRIMARY_TRIBES
from src.geo import utils as geo_utils
from src.indigenous.sovereignty import (
    generate_citations,
    print_data_acknowledgment,
)
from src.viz import charts, styles

styles.apply_mpl_style()
%matplotlib inline

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="geopandas")

print(f"Repo root : {REPO_ROOT}")
print(f"Output dir: {constants.OUTPUTS_DIR}")
print(f"Analysis run: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

Repo root : C:\Users\gekek\Documents\tfs_refactor
Output dir: C:\Users\gekek\Documents\tfs_refactor\outputs
Analysis run: 2026-03-30 18:26


In [2]:
# Data sovereignty acknowledgment
print_data_acknowledgment(
    source_keys=["census_aiannh", "usgs_wbd", "epa_ecoregions_l3", "mtbs"]
)

DATA SOVEREIGNTY ACKNOWLEDGMENT
This analysis uses data that describes Indigenous and Tribal lands,
communities, and fire histories. This project is guided by three
complementary data governance frameworks:

OCAP® — Tribal Nations own, control, access, and possess data about
  their own communities and territories.
  Reference: https://fnigc.ca/ocap-training/

CARE  — Data use must deliver Collective Benefit to Indigenous peoples,
  respect their Authority to Control, uphold Responsibility to communities,
  and center Ethics across the full data lifecycle.
  Reference: https://www.gida-global.org/care

FAIR  — Data is Findable, Accessible, Interoperable, and Reusable.
  FAIR governs technical standards; CARE and OCAP® govern the ethical
  obligations to Tribal Nations that FAIR alone does not address.
  Reference: https://www.go-fair.org/fair-principles/

We recognize that:

• Tribal Nations are sovereign governments with the right to control
  data about their own communities and terr

## Load Tribal Land Boundaries

In [3]:
# Census TIGER AIANNH 
all_tribal = loaders.load_census_aian()
all_tribal = validators.validate_geodataframe(
    all_tribal, "census_aiannh",
    required_columns=["geometry", "NAME"],
)

tribal_lands = all_tribal[
    all_tribal["NAME"].isin(PRIMARY_TRIBES)
].copy().reset_index(drop=True)

# Dissolve non-contiguous parcels by NAME
tribal_lands = tribal_lands.dissolve(by="NAME", as_index=False).reset_index(drop=True)

# Repair any invalid geometries from dissolve
tribal_lands["geometry"] = tribal_lands.geometry.apply(
    lambda g: make_valid(g) if g is not None else g
)
tribal_lands = tribal_lands[
    tribal_lands.geometry.notnull() & tribal_lands.geometry.is_valid
].copy().reset_index(drop=True)

# Compute projected centroids for distance calculations
centroids_proj = tribal_lands.to_crs("EPSG:5070").geometry.centroid
centroids_geo  = centroids_proj.to_crs(constants.CRS_GEOGRAPHIC)
tribal_lands["centroid_lon"] = centroids_geo.x
tribal_lands["centroid_lat"] = centroids_geo.y

print(f"Tribal Nations loaded: {len(tribal_lands)}")
print(tribal_lands[["NAME"]].to_string(index=False))

# Study area bounds — used for all spatial queries below
BUFFER_DEG = 1.0
bounds = tribal_lands.total_bounds
STUDY_BBOX = (
    bounds[0] - BUFFER_DEG,
    bounds[1] - BUFFER_DEG,
    bounds[2] + BUFFER_DEG,
    bounds[3] + BUFFER_DEG,
)
print(f"\nStudy bbox: {tuple(round(x, 2) for x in STUDY_BBOX)}")

Tribal Nations loaded: 10
                                  NAME
                              Cherokee
                             Chickasaw
                               Choctaw
                              Colville
                                 Creek
                           Fort Apache
Kiowa-Comanche-Apache-Fort Sill Apache
                                 Osage
                            San Carlos
                          Warm Springs

Study bbox: (np.float64(-122.82), np.float64(31.88), np.float64(-93.43), np.float64(49.48))


## Load Watersheds and Ecoregions

In [4]:
# USGS WBD HUC-8 watersheds
# Load by bbox only, server rejects combined WHERE and spatial filter.
# Filter to relevant HUC-2 regions in Python after loading.

watersheds_raw = loaders.load_usgs_wbd_huc8(
    bbox=STUDY_BBOX,
    huc2_codes=None,   # no WHERE clause — use bbox only
)
watersheds = validators.validate_geodataframe(
    watersheds_raw, "usgs_wbd",
    required_columns=["geometry", "name"],
)

# Keep only HUC-2 regions relevant to PRIMARY_TRIBES
HUC2_OF_INTEREST = ["11", "15", "17"]
watersheds = watersheds[
    watersheds["huc8"].str[:2].isin(HUC2_OF_INTEREST)
].copy().reset_index(drop=True)

# Clip to study area
study_poly = geo_utils.bbox_geodataframe(STUDY_BBOX).geometry.iloc[0]
watersheds = watersheds[watersheds.intersects(study_poly)].copy().reset_index(drop=True)

print(f"HUC-8 watersheds loaded: {len(watersheds):,}")
watersheds[["huc8", "name", "areasqkm"]].head()

HTTPError: 500 Server Error: Internal Server Error for url: https://hydro.nationalmap.gov/arcgis/rest/services/wbd/MapServer/4/query?where=1%3D1&outFields=huc8%2Cname%2Careasqkm%2Cstates&f=geojson&returnGeometry=true&resultRecordCount=2000&outSR=4326&geometry=-122.8161022638304%2C31.875138228796125%2C-93.43101644936192%2C49.48280034584955&geometryType=esriGeometryEnvelope&spatialRel=esriSpatialRelIntersects&inSR=4326&resultOffset=0

In [ ]:
# EPA Level III Ecoregions
ecoregions = loaders.load_epa_ecoregions_l3(bbox=STUDY_BBOX)
ecoregions = validators.validate_geodataframe(
    ecoregions, "epa_ecoregions_l3",
    required_columns=["geometry", "US_L3NAME"],
)

ecoregions = ecoregions[ecoregions.intersects(study_poly)].copy().reset_index(drop=True)

print(f"EPA Level III ecoregions loaded: {len(ecoregions):,}")
ecoregions[["US_L3CODE", "US_L3NAME", "NA_L2NAME"]].drop_duplicates().head(10)

## Shared Watershed Analysis

In [ ]:
def analyze_shared_watersheds(
    tribal_gdf: gpd.GeoDataFrame,
    watersheds_gdf: gpd.GeoDataFrame,
    name_col: str = "NAME",
) -> pd.DataFrame:
    """
    Identify HUC-8 watersheds intersected by two or more Tribal land units.
    Returns one row per shared watershed with the list of Tribal Nations involved.
    """
    records = []
    for _, ws in watersheds_gdf.iterrows():
        overlapping = tribal_gdf[tribal_gdf.intersects(ws.geometry)]
        if len(overlapping) >= 2:
            # Compute each Tribe's area within this watershed (Albers)
            ws_proj  = gpd.GeoSeries([ws.geometry], crs=constants.CRS_GEOGRAPHIC).to_crs("EPSG:5070")
            tribe_areas = []
            for _, tribe in overlapping.iterrows():
                intersection = tribe.geometry.intersection(ws.geometry)
                area_km2 = (
                    gpd.GeoSeries([intersection], crs=constants.CRS_GEOGRAPHIC)
                    .to_crs("EPSG:5070")
                    .area.iloc[0] / 1e6
                )
                tribe_areas.append(area_km2)

            records.append({
                "huc8":                  ws.get("huc8", ""),
                "watershed_name":        ws.get("name", ""),
                "area_sqkm":             ws.get("areasqkm", np.nan),
                "num_tribes":            len(overlapping),
                "tribe_names":           overlapping[name_col].tolist(),
                "tribal_overlap_km2":    sum(tribe_areas),
            })

    return pd.DataFrame(records).sort_values("num_tribes", ascending=False)


shared_watersheds = analyze_shared_watersheds(tribal_lands, watersheds)

print(f"Shared watersheds (≥2 Tribal Nations): {len(shared_watersheds)}")
print("=" * 60)
for _, ws in shared_watersheds.head(10).iterrows():
    print(f"\n{ws['watershed_name']} (HUC-8: {ws['huc8']})")
    print(f"  Tribes ({ws['num_tribes']}): {', '.join(ws['tribe_names'])}")
    print(f"  Tribal overlap: {ws['tribal_overlap_km2']:.0f} km²")

## Shared Ecoregion Analysis

In [ ]:
def analyze_shared_ecoregions(
    tribal_gdf: gpd.GeoDataFrame,
    ecoregions_gdf: gpd.GeoDataFrame,
    name_col: str = "NAME",
    eco_name_col: str = "US_L3NAME",
) -> pd.DataFrame:
    """
    Identify EPA Level III ecoregions shared by two or more Tribal land units.
    Tribes in the same ecoregion share fire regimes, vegetation, and
    Traditional Ecological Knowledge contexts.
    """
    # Dissolve ecoregions by name to avoid counting sub-polygons separately
    eco_dissolved = (
        ecoregions_gdf
        .dissolve(by=eco_name_col, as_index=False)
        .reset_index(drop=True)
    )

    records = []
    for _, eco in eco_dissolved.iterrows():
        overlapping = tribal_gdf[tribal_gdf.intersects(eco.geometry)]
        if len(overlapping) >= 1:
            records.append({
                "ecoregion_name": eco[eco_name_col],
                "level2_name":    eco.get("NA_L2NAME", ""),
                "num_tribes":     len(overlapping),
                "tribe_names":    overlapping[name_col].tolist(),
            })

    df = pd.DataFrame(records).sort_values("num_tribes", ascending=False)
    return df


shared_ecoregions = analyze_shared_ecoregions(tribal_lands, ecoregions)
multi_eco = shared_ecoregions[shared_ecoregions["num_tribes"] >= 2]

print(f"Ecoregions with ≥1 Tribal Nation: {len(shared_ecoregions)}")
print(f"Ecoregions shared by ≥2 Tribal Nations: {len(multi_eco)}")
print("=" * 60)
for _, eco in multi_eco.iterrows():
    print(f"\n{eco['ecoregion_name']} ({eco['level2_name']})")
    print(f"  Tribes ({eco['num_tribes']}): {', '.join(eco['tribe_names'])}")

## Build Collaboration Network

In [ ]:
def build_collaboration_network(
    tribal_gdf: gpd.GeoDataFrame,
    shared_watersheds_df: pd.DataFrame,
    shared_ecoregions_df: pd.DataFrame,
    proximity_threshold_km: float = 200,
    name_col: str = "NAME",
) -> nx.Graph:
    """
    Build a weighted collaboration network.

    Edge weights
    ------------
    Shared watershed : +2 per shared watershed
    Shared ecoregion : +1 per shared ecoregion
    Proximity        : +1 if centroids within proximity_threshold_km
    """
    G = nx.Graph()

    for _, tribe in tribal_gdf.iterrows():
        G.add_node(
            tribe[name_col],
            centroid_lon=tribe["centroid_lon"],
            centroid_lat=tribe["centroid_lat"],
        )

    def _add_edges(df, tribe_col, weight_per_share):
        for _, row in df.iterrows():
            names = row[tribe_col]
            for i in range(len(names)):
                for j in range(i + 1, len(names)):
                    a, b = names[i], names[j]
                    if not G.has_node(a) or not G.has_node(b):
                        continue
                    if G.has_edge(a, b):
                        G[a][b]["weight"] += weight_per_share
                        G[a][b]["shared_count"] += 1
                    else:
                        G.add_edge(a, b, weight=weight_per_share, shared_count=1)

    _add_edges(shared_watersheds_df[shared_watersheds_df["num_tribes"] >= 2],
               "tribe_names", 2)
    _add_edges(shared_ecoregions_df[shared_ecoregions_df["num_tribes"] >= 2],
               "tribe_names", 1)

    # Proximity edges
    tribal_proj = tribal_gdf.to_crs("EPSG:5070")
    names = tribal_gdf[name_col].tolist()
    for i, t1 in tribal_proj.iterrows():
        for j, t2 in tribal_proj.iterrows():
            if i >= j:
                continue
            dist_km = t1.geometry.centroid.distance(t2.geometry.centroid) / 1000
            if dist_km <= proximity_threshold_km:
                a, b = tribal_gdf.loc[i, name_col], tribal_gdf.loc[j, name_col]
                if G.has_edge(a, b):
                    G[a][b]["distance_km"] = dist_km
                    G[a][b]["weight"] += 1
                else:
                    G.add_edge(a, b, weight=1, shared_count=0, distance_km=dist_km)

    return G


collab_net = build_collaboration_network(
    tribal_lands,
    shared_watersheds,
    shared_ecoregions,
    proximity_threshold_km=200,
)

print("COLLABORATION NETWORK")
print(f"  Nodes (Tribal Nations)   : {collab_net.number_of_nodes()}")
print(f"  Edges (connections)      : {collab_net.number_of_edges()}")
print(f"  Network density          : {nx.density(collab_net):.3f}")
avg_deg = sum(dict(collab_net.degree()).values()) / collab_net.number_of_nodes()
print(f"  Avg connections per Tribe: {avg_deg:.1f}")

isolated = list(nx.isolates(collab_net))
if isolated:
    print(f"  Isolated (no connections): {isolated}")

In [ ]:
# Network centrality 
degree_c      = nx.degree_centrality(collab_net)
betweenness_c = nx.betweenness_centrality(collab_net, weight="weight")

centrality = pd.DataFrame([
    {
        "tribe_name":             n,
        "num_connections":        collab_net.degree(n),
        "degree_centrality":      degree_c[n],
        "betweenness_centrality": betweenness_c[n],
    }
    for n in collab_net.nodes()
]).sort_values("degree_centrality", ascending=False)

print("Most connected Tribal Nations:")
print(centrality[["tribe_name", "num_connections", "degree_centrality", "betweenness_centrality"]].round(3).to_string(index=False))

## Collaboration Opportunity Ranking

In [ ]:
def score_collaboration_opportunities(
    G: nx.Graph,
    tribal_gdf: gpd.GeoDataFrame,
    name_col: str = "NAME",
) -> pd.DataFrame:
    """
    Score each Tribal pair on a 100-point scale.

    Scoring
    -------
    Shared resources (weight) : 0–50 pts  (proportional to edge weight)
    Proximity bonus           : 0–30 pts  (closer = higher score)
    Both in same region       : 20 pts    (binary — same ecoregion cluster)
    """
    max_weight = max((d["weight"] for _, _, d in G.edges(data=True)), default=1)

    records = []
    for a, b, data in G.edges(data=True):
        resource_score  = (data["weight"] / max_weight) * 50
        dist_km         = data.get("distance_km", None)
        proximity_score = max(0, (200 - dist_km) / 200 * 30) if dist_km else 0

        records.append({
            "tribe1":           a,
            "tribe2":           b,
            "opportunity_score": round(resource_score + proximity_score, 1),
            "edge_weight":      data["weight"],
            "shared_count":     data.get("shared_count", 0),
            "distance_km":      round(dist_km, 1) if dist_km else None,
        })

    return (
        pd.DataFrame(records)
        .sort_values("opportunity_score", ascending=False)
        .reset_index(drop=True)
    )


collab_opportunities = score_collaboration_opportunities(collab_net, tribal_lands)

print("TOP 10 COLLABORATION OPPORTUNITIES")
print("=" * 70)
print(
    collab_opportunities.head(10)[
        ["tribe1", "tribe2", "opportunity_score", "edge_weight", "distance_km"]
    ].to_string(index=False)
)

## Hierarchical Clustering

In [ ]:
# Cluster Tribal Nations by geographic distance 
# Uses centroid distances in Albers Equal Area

tribal_proj = tribal_lands.to_crs("EPSG:5070").copy()
coords = np.array([
    [geom.centroid.x, geom.centroid.y]
    for geom in tribal_proj.geometry
])
dist_matrix = distance_matrix(coords, coords) / 1000  # km

Z = linkage(dist_matrix, method="ward")
n_clusters = 3
cluster_labels = fcluster(Z, n_clusters, criterion="maxclust")
tribal_lands["cluster"] = cluster_labels

print(f"Geographic clusters (k={n_clusters}):")
for c in range(1, n_clusters + 1):
    members = tribal_lands[tribal_lands["cluster"] == c]["NAME"].tolist()
    print(f"  Cluster {c}: {', '.join(members)}")

# Dendrogram
fig, ax = plt.subplots(figsize=(12, 5))
dendrogram(
    Z,
    labels=tribal_lands["NAME"].tolist(),
    ax=ax,
    color_threshold=Z[-(n_clusters - 1), 2],
    leaf_rotation=45,
)
ax.set_title("Hierarchical Clustering of Tribal Nations by Geographic Proximity",
             fontweight="bold")
ax.set_xlabel("Tribal Nation")
ax.set_ylabel("Distance (km)")
sns.despine(ax=ax)
plt.tight_layout()
charts.save_figure(fig, "outputs/figures/tribal_collaboration_dendrogram.png")
plt.show()

## Visualizations

In [ ]:
# Regional collaboration map 
import contextily as ctx

fig, ax = plt.subplots(figsize=(15, 11))

tribal_3857 = tribal_lands.to_crs(epsg=3857)
ws_3857     = watersheds.to_crs(epsg=3857)

# Watershed boundaries
ws_3857.boundary.plot(
    ax=ax, color=styles.SKY_BLUE, linewidth=0.8,
    linestyle="--", alpha=0.5, label="HUC-8 Watersheds",
)

# Tribal lands colored by cluster
cluster_colors = {
    1: styles.FIRE_ORANGE,
    2: styles.SAGE_GREEN,
    3: styles.SKY_BLUE,
}
for c, color in cluster_colors.items():
    sub = tribal_3857[tribal_3857["cluster"] == c]
    sub.plot(ax=ax, color=color, alpha=0.45, edgecolor=styles.CHARCOAL,
             linewidth=1, label=f"Cluster {c}")

# Tribal Nation labels
for _, tribe in tribal_3857.iterrows():
    cx = tribe.geometry.centroid.x
    cy = tribe.geometry.centroid.y
    ax.annotate(
        tribe["NAME"], xy=(cx, cy),
        fontsize=8, fontweight="bold", ha="center",
        bbox=dict(boxstyle="round,pad=0.2", facecolor="white", alpha=0.75),
    )

# Draw top collaboration connection lines
top_pairs = collab_opportunities.head(8)
for _, pair in top_pairs.iterrows():
    t1 = tribal_lands[tribal_lands["NAME"] == pair["tribe1"]].iloc[0]
    t2 = tribal_lands[tribal_lands["NAME"] == pair["tribe2"]].iloc[0]
    line = gpd.GeoDataFrame(
        geometry=[LineString([
            (t1["centroid_lon"], t1["centroid_lat"]),
            (t2["centroid_lon"], t2["centroid_lat"]),
        ])],
        crs=constants.CRS_GEOGRAPHIC,
    ).to_crs(epsg=3857)
    lw = pair["opportunity_score"] / 25
    line.plot(ax=ax, color="purple", linewidth=lw, alpha=0.55)

ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)
ax.set_axis_off()
ax.set_title(
    "Inter-Tribal Fire Collaboration Network\n"
    "Shared Watersheds, Ecoregions, and Top Collaboration Links",
    fontsize=13, fontweight="bold",
)
legend_elements = [
    mpatches.Patch(facecolor=c, alpha=0.5, label=f"Cluster {k}")
    for k, c in cluster_colors.items()
] + [
    plt.Line2D([0], [0], color="purple", linewidth=2, alpha=0.6, label="Collaboration links (top 8)"),
    plt.Line2D([0], [0], color=styles.SKY_BLUE, linestyle="--", label="HUC-8 Watersheds"),
]
ax.legend(handles=legend_elements, loc="lower left", fontsize=9)
plt.tight_layout()
charts.save_figure(fig, "outputs/figures/tribal_collaboration_map.png")
plt.show()

In [ ]:
# Network graph
fig, ax = plt.subplots(figsize=(14, 10))

pos = {
    n: (d["centroid_lon"], d["centroid_lat"])
    for n, d in collab_net.nodes(data=True)
}

cluster_map = dict(zip(tribal_lands["NAME"], tribal_lands["cluster"]))
node_colors = [
    cluster_colors.get(cluster_map.get(n, 1), styles.SMOKE_GRAY)
    for n in collab_net.nodes()
]
edge_widths = [
    d["weight"] * 0.8 for _, _, d in collab_net.edges(data=True)
]

nx.draw_networkx_nodes(
    collab_net, pos, node_size=600,
    node_color=node_colors, alpha=0.85, ax=ax,
)
nx.draw_networkx_edges(
    collab_net, pos, width=edge_widths,
    alpha=0.4, edge_color=styles.CHARCOAL, ax=ax,
)
nx.draw_networkx_labels(
    collab_net, pos,
    labels={n: n.split()[0] for n in collab_net.nodes()},
    font_size=8, font_weight="bold", ax=ax,
)

ax.set_title(
    "Tribal Collaboration Network\n"
    "Node color = geographic cluster | Edge width = connection strength",
    fontsize=12, fontweight="bold",
)
ax.axis("off")
plt.tight_layout()
charts.save_figure(fig, "outputs/figures/tribal_collaboration_network.png")
plt.show()

In [ ]:
# Opportunity ranking chart 
top_n = 12
top   = collab_opportunities.head(top_n)
labels = [f"{r['tribe1'].split()[0]} ↔ {r['tribe2'].split()[0]}" for _, r in top.iterrows()]

fig, ax = plt.subplots(figsize=(11, 6))
ax.barh(
    labels[::-1], top["opportunity_score"].values[::-1],
    color=styles.FIRE_ORANGE, alpha=0.85,
)
ax.set_xlabel("Opportunity Score")
ax.set_title(f"Top {top_n} Tribal Collaboration Opportunities",
             fontsize=13, fontweight="bold")
ax.set_xlim(0, 100)
sns.despine(ax=ax)
plt.tight_layout()
charts.save_figure(fig, "outputs/figures/collaboration_opportunity_ranking.png")
plt.show()

## Exports

In [ ]:
# Tabular exports
csv_exports = {
    "collaboration_opportunities.csv":  collab_opportunities,
    "network_centrality.csv":           centrality,
    "shared_watersheds.csv":            shared_watersheds.drop(columns="tribe_names", errors="ignore"),
    "shared_ecoregions.csv":            shared_ecoregions.drop(columns="tribe_names", errors="ignore"),
}
for fname, df in csv_exports.items():
    path = constants.OUTPUTS_DIR / fname
    df.to_csv(path, index=False)
    print(f"Exported → {path.relative_to(REPO_ROOT)}")

# Network edge list
edge_list = [
    {"tribe1": a, "tribe2": b, **data}
    for a, b, data in collab_net.edges(data=True)
]
edge_path = constants.OUTPUTS_DIR / "network_edges.csv"
pd.DataFrame(edge_list).to_csv(edge_path, index=False)
print(f"Exported → {edge_path.relative_to(REPO_ROOT)}")

## Summary and Findings

*(Fill in after running with real data.)*

Questions to address in the narrative:
- Which watershed basins create the strongest natural collaboration basis?
- Do the geographic clusters align with known inter-Tribal relationships or
  existing coordination frameworks (ex. intertribal councils, compacts)?
- What are the limitations? This analysis uses ecological and geographic
  proximity as proxies for collaboration potential, actual collaboration
  depends on Tribal sovereignty, governance structures, and relationships
  that no dataset can fully represent.
- How should this analysis be used? As a starting point for conversation,
  not as a prescription. Any coordination framework must be Tribal-led.

---
## References

In [ ]:
print(generate_citations(["census_aiannh", "usgs_wbd", "epa_ecoregions_l3", "mtbs"]))